In [1]:

import re

# Check whether a term is a variable
def is_variable(x):
    return isinstance(x, str) and x[0].islower()

# Convert input into a predicate structure
def parse(s):
    tokens = re.findall(r'[A-Za-z0-9_]+|[(),]', s)
    pos = [0]

    def read():
        name = tokens[pos[0]]
        pos[0] += 1

        if pos[0] < len(tokens) and tokens[pos[0]] == '(':
            pos[0] += 1
            args = []

            while True:
                args.append(read())

                if tokens[pos[0]] == ')':
                    pos[0] += 1
                    break

                pos[0] += 1

            return (name, args)

        return name

    return read()

# Apply substitutions
def apply(term, subst):
    if isinstance(term, str):
        while term in subst:
            term = subst[term]
        return term

    return (term[0], [apply(x, subst) for x in term[1]])

# Check occurs condition
def occurs(var, term, subst):
    term = apply(term, subst)

    if var == term:
        return True

    if isinstance(term, tuple):
        return any(occurs(var, x, subst) for x in term[1])

    return False

# Unification function
def unify(a, b, subst):
    a = apply(a, subst)
    b = apply(b, subst)

    if a == b:
        return True

    if is_variable(a):
        if occurs(a, b, subst):
            return False
        subst[a] = b
        return True

    if is_variable(b):
        if occurs(b, a, subst):
            return False
        subst[b] = a
        return True

    if isinstance(a, tuple) and isinstance(b, tuple):
        if a[0] != b[0] or len(a[1]) != len(b[1]):
            return False

        for x, y in zip(a[1], b[1]):
            if not unify(x, y, subst):
                return False

        return True

    return False

# Main program
e1 = input("Enter first expression: ")
e2 = input("Enter second expression: ")

a = parse(e1)
b = parse(e2)

subst = {}

if unify(a, b, subst):
    print("Unification Successful!")
    print("Substitutions:")

    for key, value in subst.items():
        print(key, "=", value)

    print("Unified Expression:", apply(a, subst))
else:
    print("Unification Failed!")


Enter first expression: Eats(x, Apple)
Enter second expression: Eats(Riya,y)
Unification Successful!
Substitutions:
x = Riya
y = Apple
Unified Expression: ('Eats', ['Riya', 'Apple'])


In [2]:

# Forward Chaining Algorithm

# Initial facts
facts = {"A", "B", "E"}

# Rules: (Premises, Conclusion)
rules = [
    (["A", "B"], "C"),
    (["C"], "D"),
    (["D", "E"], "F"),
    (["F"], "G")
]

# Goal to be achieved
goal = "G"

print("Initial Facts:", facts)
print("\nApplying Forward Chaining...\n")

# Forward Chaining process
new_fact = True

while new_fact:
    new_fact = False

    for premises, conclusion in rules:

        # Check whether all premises are true
        if all(p in facts for p in premises):

            # Add conclusion if it is not already present
            if conclusion not in facts:
                facts.add(conclusion)
                print("Derived:", conclusion)
                new_fact = True

# Display final results
print("\nFinal Facts:", facts)

if goal in facts:
    print("Goal", goal, "achieved!")
else:
    print("Goal", goal, "not achieved!")


Initial Facts: {'B', 'E', 'A'}

Applying Forward Chaining...

Derived: C
Derived: D
Derived: F
Derived: G

Final Facts: {'E', 'F', 'D', 'G', 'C', 'B', 'A'}
Goal G achieved!
